# How the synthetic point clouds are made

A walkthrough of how synthetic clouds are generated, from a YAML `process:` block to the
`data/<process>/clouds.pkl` files that every experiment trains on. The notebook has two goals:

1. **Explain.** Every stage is rebuilt in the notebook, step by step, from the repo's own code.
2. **Scrutinise.** Wherever the notebook re-implements a step, it checks the result against the
   library output or the data on disk. It also points out design choices that shape how the
   results should be read. Section 7 collects the findings.

## The pipeline at a glance

```
configs/runs/<process>/*.yaml   ──  only the `process:` block matters for generation
       │
       ▼   CloudDesign.build(process, seed=0, design, adversarial)        [design.py]
  A. draw 8000 parameter vectors θ       design RNG  = default_rng(seed)          (seed = 0)
     hold out 12.5% as "adversarial"     holdout RNG = default_rng(seed + 100000)
       │
       ▼   CloudDesign.generate()
  B. cloud i  = Process(**θ_i).sample(region=[0,1]², seed = i)              i = 0 … 6999
     adv. j   = Process(**θ_j).sample(region=[0,1]², seed = 100000 + j)    j = 0 … 999
       │
       ▼   scripts/generate.py
  C. data/<process>/{clouds.pkl, adversarial_clouds.pkl, cloud_generation_manifest.yaml}
       │
       ▼   scripts/build_classification_data.py   (classification task only)
     data/classification/   (seed → source_dir_index·10⁶ + seed,  aniso_thomas labelled "thomas")
```

| § | Content |
|---|---|
| 1 | Setup: the window, what a "cloud" record is |
| 2 | **Stage A**: from a config to parameter vectors (the reparametrisation and the constraint) |
| 3 | **Stage B**: the Neyman–Scott engine, one random draw at a time, and the edge buffer |
| 4 | The families: kernels, Matérn cluster, anisotropic Thomas, nested Thomas, Strauss (MCMC) |
| 5 | **Parameter-space pictures**: where the design puts its clouds, in 3-D |
| 6 | **Stage C**: seeds, the adversarial holdout, cross-process coupling, point counts |
| 7 | Scrutiny summary |

**Runtime.** Most cells take a few seconds. The slowest is the Strauss MCMC convergence
check in §4.5, which takes a couple of minutes. The checks against stored data need
`data/<process>/clouds.pkl`. If those files are missing, the cells print a note and skip.

## 0. Setup

In [ ]:
import sys, math, pickle, inspect, itertools, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml
from matplotlib.collections import LineCollection
from scipy.stats import norm, poisson, ks_2samp, spearmanr
from scipy.spatial import cKDTree

# repo root = first ancestor holding src/cloudforger (works from notebooks/ or the repo root)
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "cloudforger").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from cloudforger.core.region import Box
from cloudforger.data_generation.design import CloudDesign, apply_derived, passes
from cloudforger.data_generation.point_processes import (
    REGISTRY, ThomasProcess, MaternClusterProcess, NestedThomasProcess,
    AnisotropicThomasProcess, StraussProcess,
    GaussianKernel, BallKernel, AnisotropicGaussianKernel,
)
from cloudforger.data_generation.point_processes.gibbs import _adaptive_steps

try:
    from IPython.display import display
except ImportError:          # headless smoke test
    display = print

try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False

DATA = ROOT / "data"
WINDOW = Box(low=np.zeros(2), high=np.ones(2))      # the observation window W = [0,1]^2, |W| = 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("repo root:", ROOT, "| plotly available:", HAVE_PLOTLY)

The five families in the headline experiments, each paired with a representative run config.
All configs for one process share the same `process:` block, except
`nested_thomas_vihrs.yaml`, `error_floor.yaml` and `mincontrast*.yaml` (see §7).

In [ ]:
CONFIGS = {
    "thomas":         "configs/runs/thomas/thomas_pi_multik_k5k10k15.yaml",
    "matern_cluster": "configs/runs/matern_cluster/matern_cluster_pi_multik_k5k10k15.yaml",
    "aniso_thomas":   "configs/runs/aniso_thomas/aniso_thomas_pi_multik_k5k10k15.yaml",
    "nested_thomas":  "configs/runs/nested_thomas/pi_multik.yaml",
    "strauss":        "configs/runs/strauss/strauss_pi_multik_k5k10k15.yaml",
}
_RAW = {name: yaml.safe_load(open(ROOT / path)) for name, path in CONFIGS.items()}

def process_block(name):
    """The `process:` block -- the only part of a run config that scripts/generate.py reads."""
    return _RAW[name]["process"]

def targets_of(name):
    """(target_label_names, log_label_names) -- what the regression head is trained on."""
    return _RAW[name]["target_label_names"], set(_RAW[name].get("log_label_names", []))

def make_process(name, theta):
    """Exactly what design.generate_clouds_for_design does: keep only the keys the
    constructor accepts (sampling coordinates such as K, EN, c are silently dropped)."""
    cls = REGISTRY.get(name)
    ok = set(inspect.signature(cls.__init__).parameters) - {"self"}
    return cls(**{k: v for k, v in theta.items() if k in ok})

def rebuild_design(name):
    blk = process_block(name)
    return CloudDesign.build(blk["name"], seed=blk["seed"], design=blk["design"],
                             adversarial=blk.get("adversarial"))

t0 = time.time()
DESIGNS = {name: rebuild_design(name) for name in CONFIGS}      # parameter vectors only; no simulation
print(f"rebuilt {len(DESIGNS)} designs in {time.time() - t0:.1f}s")
for name, d in DESIGNS.items():
    print(f"  {name:15s} train/test vectors: {len(d.train_test_vectors):5d}   adversarial: {len(d.adversarial_vectors):5d}")

In [ ]:
_RECORDS = {}

def load_records(name, adversarial=False):
    """Stored clouds: a list of dict records (core.records.cloud_to_record). None if absent."""
    key = (name, adversarial)
    if key not in _RECORDS:
        path = DATA / name / ("adversarial_clouds.pkl" if adversarial else "clouds.pkl")
        if not path.exists():
            print(f"[skip] {path} not found")
            return None
        with open(path, "rb") as f:
            _RECORDS[key] = pickle.load(f)
    return _RECORDS[key]

def show_cloud(ax, pts, title="", s=2, color="k", alpha=0.8):
    ax.scatter(pts[:, 0], pts[:, 1], s=s, color=color, alpha=alpha, linewidths=0)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=8)

## 1. What a "cloud" is

Every cloud is one realisation of a spatial point process observed in the fixed **unit square**
$W=[0,1]^2$ (`design.default_region`). The cloud's parameters vary between clouds; the window
never does. That makes intensities (points per unit area) and point counts interchangeable,
since $|W| = 1$.

On disk a cloud is a plain dict. `params` is the process object's own `.params` property, not
the design vector, so it holds the *constructor* parameters (plus diagnostics such as
`edge_buffer` and `c1`) but not the sampling coordinates `K, EN, c` introduced in §2.

In [ ]:
recs = load_records("thomas")
if recs is not None:
    r = recs[0]
    for k, v in r.items():
        print(f"{k:10s}", v if k != "points" else f"ndarray {v.shape}, e.g. {v[:2].round(3).tolist()} ...")

## 2. Stage A: from a config to parameter vectors

### 2.1 The Thomas design block

Here is the whole generation recipe for the Thomas process:

In [ ]:
print(yaml.safe_dump(process_block("thomas"), sort_keys=False))

The Thomas process has three parameters: parent intensity $\kappa$, mean offspring per parent
$\mu$, and Gaussian cluster scale $\sigma$. The design does **not** sample those directly.
It samples three *interpretable* coordinates, each **log-uniformly** (so every factor-of-2
interval is equally likely), and then derives $(\kappa,\mu,\sigma)$:

| sampled | meaning | derived |
|---|---|---|
| $K\in[15,120]$ | parents per unit area | $\kappa = K$ |
| $EN\in[150,800]$ | **expected number of points in the window**, $\kappa\mu$ | $\mu = EN/K$ |
| $c\in[0.1,0.9]$ | **overlap index**: cluster diameter $2\sigma$ measured in units of the typical parent spacing $1/\sqrt{\kappa}$ | $\sigma = c/(2\sqrt{K})$ |

followed by one rejection constraint, $\mu \ge 2.5$ (a "cluster" of 1–2 points is not a cluster).

**Why reparametrise?** Independent boxes on $(\kappa,\mu,\sigma)$ would waste much of the budget
on clouds that are either huge or have no visible clustering (for example small $\kappa$ with
large $\sigma$, where the clusters overlap into a Poisson-looking haze). The pair $(EN, c)$
directly controls the two things that decide what a cloud *looks like*: how many points it has,
and how separable its clusters are. $c \ll 1$ gives tight, isolated clusters. $c \to 1$ gives
neighbouring clusters that touch.

Mechanically (`design.build_param_vectors`), the design repeats "draw $(K,EN,c)$ →
`apply_derived` (evaluates the `derived:` expressions in order) → `passes` (evaluates the
`constraints:`)" until 8000 vectors are accepted. All draws come from one `default_rng(seed=0)`
stream.

### 2.2 Does the rebuilt design match what is on disk?

The design is deterministic given the seed, so rebuilding it here should reproduce the stored
records exactly. The check covers (a) every target parameter of every cloud, (b) the seed
scheme, and (c) a few clouds **re-simulated from scratch and compared bit for bit**. That
confirms the code in this checkout is the code that made the data.

In [ ]:
def check_against_disk(name, n_resim=3):
    d = DESIGNS[name]
    keys, _ = targets_of(name)
    for adv, design, seed0 in [(False, d.train_test_design, d.base_seed),
                               (True, d.adversarial_design, d.base_seed + d.adversarial_seed_offset)]:
        recs = load_records(name, adversarial=adv)
        if recs is None:
            return
        tag = "adversarial" if adv else "train/test "
        assert len(recs) == len(design), (len(recs), len(design))
        rebuilt = np.array([[make_process(name, v).params[k] for k in keys] for v in design])
        stored = np.array([[r["params"][k] for k in keys] for r in recs])
        max_rel = np.max(np.abs(rebuilt - stored) / np.maximum(np.abs(stored), 1e-12))
        seeds_ok = all(r["seed"] == seed0 + i for i, r in enumerate(recs))
        idx = np.random.default_rng(1).choice(len(design), n_resim, replace=False)
        bit_ok = all(np.array_equal(make_process(name, design[i]).sample(region=WINDOW, seed=seed0 + i).points,
                                    recs[i]["points"]) for i in idx)
        print(f"{name:15s} {tag}  n={len(recs):5d}  max rel. param diff={max_rel:.1e}  "
              f"seeds = {seed0}+i: {seeds_ok}  re-simulated {n_resim} clouds bit-identical: {bit_ok}")

for name in CONFIGS:
    check_against_disk(name)

### 2.3 The constraint cuts a corner off the box

$\mu = EN/K \ge 2.5$ is the half-plane $EN \ge 2.5K$ in the $(K, EN)$ plane. It removes the
corner with many parents and few points. Two consequences:

* the accepted $K$ and $EN$ are **no longer independent** (positively correlated), and
* the accepted $K$ marginal is **no longer log-uniform**: large $K$ is under-represented.

In [ ]:
d = DESIGNS["thomas"]
K = np.array([v["K"] for v in d.train_test_vectors + d.adversarial_vectors])
EN = np.array([v["EN"] for v in d.train_test_vectors + d.adversarial_vectors])

# acceptance rate of the raw box, by Monte Carlo on the config's own ranges
rng = np.random.default_rng(123)
rngs = process_block("thomas")["design"]["random"]["ranges"]
Kb = np.exp(rng.uniform(np.log(rngs["K"]["low"]), np.log(rngs["K"]["high"]), 400_000))
ENb = np.exp(rng.uniform(np.log(rngs["EN"]["low"]), np.log(rngs["EN"]["high"]), 400_000))
print(f"acceptance rate of mu >= 2.5: {np.mean(ENb / Kb >= 2.5):.3f}")
print(f"corr(log K, log EN) among accepted vectors: {np.corrcoef(np.log(K), np.log(EN))[0, 1]:+.3f}  (0 before the cut)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
ax.scatter(K, EN, s=1, alpha=0.3, color="C0", rasterized=True, label="accepted design vectors")
kk = np.geomspace(15, 120, 100)
ax.plot(kk, 2.5 * kk, "r-", lw=1.5, label=r"$EN = 2.5K$  ($\mu = 2.5$)")
ax.fill_between(kk, 150, np.clip(2.5 * kk, 150, 800), color="r", alpha=0.12, label="rejected")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("K  (parents / unit area)"); ax.set_ylabel("EN  (expected points)")
ax.set_title("Thomas design in the sampling plane (K, EN)"); ax.legend(fontsize=7, loc="upper left")
ax = axes[1]
bins = np.geomspace(15, 120, 25)
ax.hist(Kb, bins=bins, density=True, histtype="step", color="0.5", label="raw log-uniform box")
ax.hist(K, bins=bins, density=True, histtype="step", color="C0", lw=1.5, label="after the cut")
ax.set_xscale("log"); ax.set_xlabel("K"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("marginal of K: large K thinned out")
plt.tight_layout(); plt.show()

## 3. Stage B: the Neyman–Scott engine

Thomas, Matérn cluster, anisotropic Thomas and nested Thomas are all subclasses of
`NeymanScottProcess` (`point_processes/neyman_scott.py`). They differ only in the **offspring
kernel** and, for nested Thomas, in where the parents come from. The algorithm:

1. **Grow the window** by an edge buffer $b$: $W^+ = [-b, 1+b]^2$.
2. **Parents**: $N_p \sim \text{Poisson}(\kappa\,|W^+|)$, placed uniformly in $W^+$.
3. **Offspring counts**: each parent independently gets $N_j \sim \text{Poisson}(\mu)$ children.
4. **Offspring positions**: child = parent + displacement drawn from the kernel
   (Thomas: $\mathcal N(0, \sigma^2 I_2)$).
5. **Clip**: keep only children inside $W$. Parents are never part of the cloud.

### 3.1 A faithful step-by-step replica

`ns_steps` below reruns `NeymanScottProcess._sample_points` but keeps every intermediate. It
makes **the same RNG calls in the same order**, so its final points must be bit-identical to the
library's `process.sample(...)`. The cell asserts this. The replica also recurses into the
meta-process for nested Thomas (§4.4).

In [ ]:
def ns_steps(proc, rng, region=WINDOW):
    """NeymanScottProcess._sample_points, with every intermediate kept."""
    d = region.dimension
    expanded = region.expanded(proc.edge_buffer)
    out = {"region": region, "expanded": expanded}
    if isinstance(proc, NestedThomasProcess):
        # parents are themselves a Thomas process, simulated on W+ and clipped to W+
        out["meta"] = ns_steps(proc.meta_process, rng, expanded)
        parents = out["meta"]["points"]
    else:
        n_par = int(rng.poisson(proc.parent_intensity * expanded.volume))
        parents = expanded.sample_uniform(n_par, rng) if n_par else np.empty((0, d))
    counts, offsets = np.zeros(0, dtype=int), np.empty((0, d))
    if len(parents):
        counts = np.asarray(proc.offspring_count_sampler(len(parents), rng), dtype=int)
        if counts.sum():
            offsets = np.asarray(proc.kernel.sample(int(counts.sum()), d, rng), dtype=float)
    parent_idx = np.repeat(np.arange(len(parents)), counts)
    offspring = parents[parent_idx] + offsets if len(offsets) else np.empty((0, d))
    inside = region.contains(offspring) if len(offspring) else np.zeros(0, dtype=bool)
    out.update(parents=parents, counts=counts, parent_idx=parent_idx, offsets=offsets,
               offspring=offspring, inside=inside, points=offspring[inside])
    return out

def steps(proc, seed):
    return ns_steps(proc, np.random.default_rng(seed))

# a readable Thomas example, pushed through the config's own `derived:` expressions
DERIVED_T = process_block("thomas")["design"]["random"]["derived"]
theta_demo = apply_derived({"K": 25.0, "EN": 300.0, "c": 0.25}, DERIVED_T)
proc_demo = make_process("thomas", theta_demo)
SEED_DEMO = 3
st = steps(proc_demo, SEED_DEMO)

lib_points = proc_demo.sample(region=WINDOW, seed=SEED_DEMO).points
assert np.array_equal(st["points"], lib_points), "replica diverged from the library!"
print("replica == library output, bit for bit")
print({k: round(v, 4) for k, v in proc_demo.params.items()})
print(f"|W+| = {st['expanded'].volume:.3f}   E[parents] = kappa|W+| = {proc_demo.parent_intensity * st['expanded'].volume:.1f}"
      f"   drawn: {len(st['parents'])}")
print(f"offspring generated: {len(st['offspring'])}   kept in W: {len(st['points'])}   (EN = {theta_demo['EN']:.0f})")

In [ ]:
def draw_windows(ax, st):
    e = st["expanded"]
    ax.add_patch(plt.Rectangle(e.low, *(e.high - e.low), fill=False, ls="--", ec="0.5", lw=1))
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, ec="k", lw=1.2))
    pad = 0.02 + (e.high[0] - 1)
    ax.set_xlim(-pad, 1 + pad); ax.set_ylim(-pad, 1 + pad); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

P, cnt, idx, off = st["parents"], st["counts"], st["parent_idx"], st["offspring"]
col = plt.cm.tab20(np.arange(len(P)) % 20)

fig, axes = plt.subplots(2, 3, figsize=(13, 8.6))
ax = axes[0, 0]; draw_windows(ax, st)
ax.scatter(P[:, 0], P[:, 1], marker="x", color="r", s=30)
ax.set_title(f"1-2. {len(P)} parents ~ Poisson(κ|W⁺|), uniform on W⁺\n(dashed: W⁺, buffer b = {proc_demo.edge_buffer:.3f} ≈ 3.72σ)", fontsize=9)

ax = axes[0, 1]; draw_windows(ax, st)
ax.scatter(P[:, 0], P[:, 1], s=8 + 6 * cnt, color=col, alpha=0.7)
for (x, y), n in zip(P, cnt):
    ax.text(x, y, str(n), fontsize=6, ha="center", va="center")
ax.set_title(f"3. each parent gets N ~ Poisson(μ = {proc_demo.params['mean_offspring']:.1f}) children", fontsize=9)

ax = axes[0, 2]; draw_windows(ax, st)
ax.add_collection(LineCollection(np.stack([P[idx], off], axis=1), colors=col[idx], lw=0.4, alpha=0.6))
ax.scatter(off[:, 0], off[:, 1], s=3, color=col[idx])
ax.set_title(f"4. child = parent + N(0, σ²I),  σ = {proc_demo.params['cluster_scale']:.3f}", fontsize=9)

ax = axes[1, 0]; draw_windows(ax, st)
ins = st["inside"]
ax.scatter(off[~ins, 0], off[~ins, 1], s=3, color="0.75", label=f"discarded ({(~ins).sum()})")
ax.scatter(off[ins, 0], off[ins, 1], s=3, color=col[idx][ins], label=f"kept ({ins.sum()})")
ax.legend(fontsize=7, loc="upper right"); ax.set_title("5. clip to W: parents and outside children dropped", fontsize=9)

show_cloud(axes[1, 1], st["points"], f"what the model sees: {len(st['points'])} unlabeled points", s=3)

ax = axes[1, 2]
ks = np.arange(0, cnt.max() + 5)
ax.bar(ks, np.bincount(cnt, minlength=len(ks))[:len(ks)] / len(cnt), color="C0", alpha=0.6, label="this cloud")
ax.plot(ks, poisson.pmf(ks, proc_demo.params["mean_offspring"]), "ko-", ms=3, label="Poisson(μ) pmf")
ax.set_xlabel("children per parent"); ax.legend(fontsize=7); ax.set_title("offspring count distribution", fontsize=9)
plt.tight_layout(); plt.show()

Two facts about the output that matter downstream:

* **The point count is random, and over-dispersed.** $n$ is a Poisson number of parents, each
  contributing a Poisson number of children, so $\mathrm{Var}(n) > \mathbb E[n] = EN$. The model
  observes $n$ and gets a noisy estimate of the product $\kappa\mu$ almost for free. The hard
  part is splitting that product into $\kappa$ and $\mu$.
* **$EN$ is exactly $\mathbb E[n]$** only because of the edge buffer. The next cell tests that.

### 3.2 Why the edge buffer exists (and what happens without it)

A point near the edge of $W$ can have its parent *outside* $W$. If parents were only placed in
$W$ itself, the edges would be under-populated. Clouds would then have a density that falls off
at the border, and fewer points than $EN$.

For Gaussian kernels the buffer is `GaussianKernel.support_radius(eps=1e-4)`
$= \sigma\,\Phi^{-1}(1-10^{-4}) \approx 3.72\sigma$: a parent further out than that reaches the
window with per-axis probability below $10^{-4}$. For the uniform-disk (Matérn) kernel the buffer
is exactly the disk radius. The cell below averages 400 clouds and compares the empirical
intensity profile across $x$ with the buffer and with `edge_buffer=0`.

In [ ]:
theta_e = apply_derived({"K": 25.0, "EN": 300.0, "c": 0.6}, DERIVED_T)   # wide clusters: sigma = 0.06
with_buf = make_process("thomas", theta_e)
no_buf = ThomasProcess(theta_e["parent_intensity"], theta_e["mean_offspring"], theta_e["cluster_scale"], edge_buffer=0.0)
N_REP = 400
xs = {lab: [p.sample(region=WINDOW, seed=s).points for s in range(N_REP)] for lab, p in
      [("default buffer", with_buf), ("edge_buffer = 0", no_buf)]}

bins = np.linspace(0, 1, 41)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for lab, clouds in xs.items():
    allx = np.concatenate([c[:, 0] for c in clouds])
    h, _ = np.histogram(allx, bins=bins)
    axes[0].step(bins[:-1], h / (N_REP * np.diff(bins)), where="post", label=lab)
    axes[1].hist([len(c) for c in clouds], bins=30, histtype="step", label=f"{lab}: mean n = {np.mean([len(c) for c in clouds]):.0f}")
axes[0].axhline(theta_e["EN"], color="k", ls=":", label="EN (target intensity)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("points per unit area"); axes[0].legend(fontsize=7)
axes[0].set_title(f"intensity profile across the window (σ = {theta_e['cluster_scale']:.3f})")
axes[1].axvline(theta_e["EN"], color="k", ls=":"); axes[1].set_xlabel("n points"); axes[1].legend(fontsize=7)
axes[1].set_title("point counts over 400 clouds")
plt.tight_layout(); plt.show()
print(f"buffer used: {with_buf.edge_buffer:.4f} = {with_buf.edge_buffer / theta_e['cluster_scale']:.2f} sigma")

## 4. The families

### 4.1 Three offspring kernels, one notion of "cluster width"

| family | kernel (`kernels.py`) | shape parameter(s) from the design |
|---|---|---|
| Thomas | `GaussianKernel(σ)`: $\mathcal N(0,\sigma^2 I)$ | $\sigma = c/(2\sqrt K)$ |
| Matérn cluster | `BallKernel(R)`: uniform on the disk of radius $R$ | $R = c/\sqrt K$ |
| anisotropic Thomas | `AnisotropicGaussianKernel(σ₁, σ₂, θ)`: an ellipse rotated by θ | $\sigma_{1,2} = \sigma\,a^{\pm 1/2}$, aspect $a\in[1,4]$, $\theta\in[0,\pi)$ |

The three are tied together by the **per-axis RMS displacement** $r$. A uniform disk of radius
$R$ has $\mathbb E[X^2] = R^2/4$, so $R = 2\sigma$ gives the same $r = \sigma$ as the Gaussian.
The design uses the same $c$ for both, which makes "same $c$" mean "same cluster width in the RMS
sense". The anisotropic kernel keeps the *geometric mean* $\sqrt{\sigma_1\sigma_2} = \sigma$
fixed. So at equal $(K, EN, c)$ the families differ only in cluster **shape**: Gaussian tails,
a hard-edged disk, or an ellipse.

In [ ]:
rng = np.random.default_rng(0)
sig, a_, th_ = 1.0, 3.0, np.pi / 6
kern = {
    "Gaussian σ=1  (Thomas)": GaussianKernel(sig),
    "uniform disk R=2σ  (Matérn)": BallKernel(2 * sig),
    f"ellipse a={a_:.0f}, θ=π/6  (aniso Thomas)": AnisotropicGaussianKernel(sig * a_ ** 0.5, sig / a_ ** 0.5, th_),
}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (lab, k) in zip(axes, kern.items()):
    z = k.sample(20_000, 2, rng)
    ax.hexbin(z[:, 0], z[:, 1], gridsize=45, extent=(-5, 5, -5, 5), cmap="Greys", mincnt=1)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect("equal")
    rms = np.sqrt((z ** 2).mean(0))
    gm = np.linalg.det(np.cov(z.T)) ** 0.25
    ax.set_title(f"{lab}\nper-axis RMS = ({rms[0]:.2f}, {rms[1]:.2f});  det(Σ)^¼ = {gm:.2f}\n"
                 f"edge buffer = {k.support_radius():.2f}", fontsize=8)
plt.tight_layout(); plt.show()

### 4.2 Thomas and Matérn cluster at the same parameters

The same $(K, EN, c)$ and the same seed, for three overlap indices. The Matérn clusters have
hard edges and a flat interior. The Thomas clusters have a dense core and a diffuse halo. As
$c$ grows, both approach a structureless, Poisson-like pattern, which is why large $c$ is the
hard end of the design.

*Look closely*: the clusters sit in almost the **same places** in each column's two panels.
That comes from the shared seed, not from the physics. §6.3 measures it.

In [ ]:
DERIVED_M = process_block("matern_cluster")["design"]["random"]["derived"]
fig, axes = plt.subplots(2, 3, figsize=(11, 7.4))
for j, c in enumerate([0.15, 0.4, 0.85]):
    u = {"K": 30.0, "EN": 400.0, "c": c}
    for i, (name, der) in enumerate([("thomas", DERIVED_T), ("matern_cluster", DERIVED_M)]):
        p = make_process(name, apply_derived(u, der))
        pts = p.sample(region=WINDOW, seed=11).points
        extra = f"σ = {p.params['cluster_scale']:.3f}" if name == "thomas" else f"R = {p.params['cluster_radius']:.3f}"
        show_cloud(axes[i, j], pts, f"{name}  c = {c}  ({extra})  n = {len(pts)}", s=3)
plt.tight_layout(); plt.show()

### 4.3 Anisotropic Thomas

The same design as Thomas with two extra axes: aspect $a \sim U[1,4]$ (linear) and orientation
$\theta \sim U[0,\pi)$. The ellipse is $\pi$-periodic, so $[0,\pi)$ covers each orientation once.
In the classification task these clouds are **merged into the `thomas` class**, so they
broaden what "Thomas" looks like to the classifier. A third of them have $a < 2$, which is
close to isotropic.

In [ ]:
DERIVED_A = process_block("aniso_thomas")["design"]["random"]["derived"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, (a, th) in zip(axes, [(1.0, 0.0), (2.0, np.pi / 4), (4.0, np.pi / 4), (4.0, 3 * np.pi / 4)]):
    u = {"K": 20.0, "EN": 400.0, "c": 0.3, "cluster_aspect": a, "cluster_theta": th}
    p = make_process("aniso_thomas", apply_derived(u, DERIVED_A))
    show_cloud(ax, p.sample(region=WINDOW, seed=5).points,
               f"aspect {a:.0f}, θ = {th / np.pi:.2f}π\nσ₁ = {p.params['cluster_sigma_1']:.3f}, σ₂ = {p.params['cluster_sigma_2']:.3f}", s=3)
plt.tight_layout(); plt.show()

### 4.4 Nested (two-level) Thomas

`NestedThomasProcess` is a Neyman–Scott process whose parents are *themselves* a Thomas process
(the `meta_process`):

1. **grandparents** $\sim$ Poisson($\kappa$), uniform on the doubly-expanded window $W^{++}$;
2. each grandparent gets Poisson($\mu_1$) **parents**, displaced by $\mathcal N(0,\sigma_1^2 I)$,
   clipped to $W^+$;
3. each parent gets Poisson($\mu$) **children**, displaced by $\mathcal N(0,\sigma_2^2 I)$,
   clipped to $W$.

The design samples five coordinates log-uniformly, in the same spirit as the Thomas design:

| sampled | derived | meaning |
|---|---|---|
| $K\in[15,120]$ | $\kappa$ = `meta_parent_intensity` | grandparents per unit area |
| $\mu_1\in[1.5,7]$ | `meta_offspring` | parents per grandparent |
| $EN\in[150,800]$ | $\mu = EN/(K\mu_1)$ = `mean_offspring` | expected points in W ($=\kappa\mu_1\mu$) |
| $c_1\in[0.1,0.9]$ | $\sigma_1 = c_1/(2\sqrt K)$ | overlap index of the super-clusters |
| $c_2\in[0.03,0.35]$ | $\sigma_2 = c_2/(2\sqrt{K\mu_1})$ | overlap index of the clusters, relative to the *parent* spacing $1/\sqrt{\kappa\mu_1}$ |

Constraints: $\mu \ge 2.5$ and $3 \le \sigma_1/\sigma_2 \le 12$. The second one is a
**scale-separation** condition. Since $\sigma_1/\sigma_2 = (c_1/c_2)\sqrt{\mu_1}$, it requires the
two levels of clustering to be visibly distinct without being so far apart that each
super-cluster collapses to one blob.

Naming trap: in the stored `params` and in `target_label_names`, **`parent_intensity` is the
grandparent intensity $\kappa$** (`NeymanScottProcess.params` reports the constructor's
`parent_intensity`, which `NestedThomasProcess` sets to `meta_parent_intensity`).

In [ ]:
DERIVED_N = process_block("nested_thomas")["design"]["random"]["derived"]
CONSTR_N = process_block("nested_thomas")["design"]["random"]["constraints"]
theta_n = apply_derived({"K": 20.0, "mu1": 4.0, "EN": 400.0, "c1": 0.5, "c2": 0.15}, DERIVED_N)
print("inside the design constraints:", passes(theta_n, CONSTR_N),
      f"| sigma1/sigma2 = {theta_n['meta_cluster_scale'] / theta_n['cluster_scale']:.1f}")
proc_n = make_process("nested_thomas", theta_n)
stn = steps(proc_n, seed=4)
assert np.array_equal(stn["points"], proc_n.sample(region=WINDOW, seed=4).points)
print("nested replica == library output, bit for bit")

meta = stn["meta"]
G = meta["parents"]                                   # grandparents (on W++)
gp_of_parent = meta["parent_idx"][meta["inside"]]     # grandparent index of each kept parent
gp_of_child = gp_of_parent[stn["parent_idx"]]
colg = plt.cm.tab20(np.arange(len(G)) % 20)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
ax = axes[0]; draw_windows(ax, meta)
ax.scatter(G[:, 0], G[:, 1], marker="*", s=80, color=colg, ec="k", lw=0.4)
ax.set_title(f"level 0: {len(G)} grandparents on W⁺⁺\nκ = {theta_n['meta_parent_intensity']:.0f}", fontsize=9)
ax = axes[1]; draw_windows(ax, meta)
ax.scatter(G[:, 0], G[:, 1], marker="*", s=40, color=colg, alpha=0.4)
ax.scatter(stn["parents"][:, 0], stn["parents"][:, 1], s=12, color=colg[gp_of_parent], ec="k", lw=0.3)
e1 = stn["expanded"]
ax.add_patch(plt.Rectangle(e1.low, *(e1.high - e1.low), fill=False, ls=":", ec="C1", lw=1.2))
ax.set_title(f"level 1: {len(stn['parents'])} parents (clipped to W⁺)\nμ₁ = {theta_n['meta_offspring']:.1f}, σ₁ = {theta_n['meta_cluster_scale']:.3f}", fontsize=9)
ax = axes[2]; draw_windows(ax, meta)
ins = stn["inside"]
ax.scatter(stn["offspring"][ins, 0], stn["offspring"][ins, 1], s=2, color=colg[gp_of_child][ins])
ax.set_title(f"level 2: children, coloured by grandparent\nμ = {theta_n['mean_offspring']:.1f}, σ₂ = {theta_n['cluster_scale']:.4f}", fontsize=9)
show_cloud(axes[3], stn["points"], f"what the model sees: n = {len(stn['points'])}", s=2)
plt.tight_layout(); plt.show()

### 4.5 Strauss: a different mechanism entirely (Gibbs process, MCMC)

The Strauss process is **repulsive**, not clustered, and has no constructive recipe. It is
defined by a density with respect to a unit-rate Poisson process,

$$ f(\mathbf x) \;\propto\; \beta^{\,n(\mathbf x)}\,\gamma^{\,S_R(\mathbf x)}, \qquad S_R = \#\{\text{pairs closer than } R\}, $$

where $\gamma\in[0,1]$ penalises close pairs ($\gamma=1$ is Poisson($\beta$), $\gamma\to 0$
approaches a hard-core process). The normalising constant is intractable, so
`gibbs._birth_death_mh` samples it with **birth–death Metropolis–Hastings** (Geyer & Møller 1994):

* with prob. ½ propose a **birth** at a uniform $u$; accept with $\min\{1,\ \beta\gamma^{t(u)}|W|/(n+1)\}$;
* with prob. ½ propose the **death** of a uniformly chosen point; accept with $\min\{1,\ n/(\beta\gamma^{t}|W|)\}$;

where $t$ is the number of current points within $R$. Implementation details worth knowing:

* The chain runs on $W$ grown by $2R$ and is clipped back to $W$ at the end (Vihrs' convention).
* It **starts from a Poisson($\beta$) pattern**, which is *over*-dense whenever $\gamma<1$.
* It runs a **fixed** `max(15000, 40·β|W⁺|)` steps (about 40 proposals per expected point).
  No convergence diagnostic is computed.
* **$\beta$ is an activity, not an intensity.** The realised intensity is below $\beta$, and far
  below it when $\gamma$ is small and $R$ is large.

The design is a plain box: $\beta\sim$ log-U[200, 900], $\gamma\sim$ U[0.05, 0.95],
$R\sim$ U[0.005, 0.05], with no derived quantities and no constraints.

`strauss_trace` below copies the sampler (same RNG calls, same order), additionally recording
$n(t)$. It is checked bit for bit against the library.

In [ ]:
def strauss_trace(proc, seed, n_steps=None, region=WINDOW, every=100):
    """gibbs._birth_death_mh for a StraussProcess, recording (step, n in W+, n in W)."""
    rng = np.random.default_rng(seed)
    sim = region.expanded(proc.margin_factor * proc.radius)
    if n_steps is None:
        n_steps = _adaptive_steps(proc.n_steps, proc.beta * sim.volume)
    low = np.asarray(sim.low, dtype=float); span = np.asarray(sim.high, dtype=float) - low
    d, vol = sim.dimension, float(sim.volume)
    log_vol, log_beta, r2 = math.log(vol), math.log(proc.beta), proc.radius ** 2
    log_gamma = -math.inf if proc.gamma <= 0.0 else math.log(proc.gamma)

    n = int(rng.poisson(proc.beta * vol))
    pts = np.empty((n + n_steps + 1, d))
    if n:
        pts[:n] = low + span * rng.random((n, d))
    coin_birth = rng.random(n_steps) < 0.5
    u_unit = rng.random((n_steps, d))
    death_u = rng.random(n_steps)
    with np.errstate(divide="ignore"):
        log_accept = np.log(rng.random(n_steps))
    trace = []
    for k in range(n_steps):
        if k % every == 0:
            trace.append((k, n, int(region.contains(pts[:n]).sum())))
        if coin_birth[k]:
            u = low + span * u_unit[k]
            t = 0 if n == 0 else int(np.count_nonzero(((pts[:n] - u) ** 2).sum(1) <= r2))
            pen = 0.0 if t == 0 else t * log_gamma
            if log_accept[k] < log_beta + pen + log_vol - math.log(n + 1):
                pts[n] = u; n += 1
        elif n > 0:
            i = int(death_u[k] * n)
            p = pts[i].copy()
            t = int(np.count_nonzero(((pts[:n] - p) ** 2).sum(1) <= r2)) - 1
            pen = 0.0 if t == 0 else t * log_gamma
            if log_accept[k] < math.log(n) - log_beta - pen - log_vol:
                pts[i] = pts[n - 1]; n -= 1
    final = pts[:n].copy()
    return final[region.contains(final)], np.array(trace), n_steps

strong = dict(beta=900.0, gamma=0.05, radius=0.05)          # the hardest corner of the design box
proc_s = StraussProcess(**strong)
pts_s, _, steps_default = strauss_trace(proc_s, seed=0)
assert np.array_equal(pts_s, proc_s.sample(region=WINDOW, seed=0).points)
print(f"Strauss replica == library output, bit for bit   (default n_steps = {steps_default})")

**Convergence check.** Run the chain 10× longer than the default, from the same start, and ask
whether $n$ has settled by the default cut (the dashed line). Then compare summary statistics
(points in $W$, close pairs $S_R$) between default-length and 10×-length chains over several
seeds, in three corners of the design. If the default budget is enough, the two should agree
within noise. *This is the slowest cell (about 1–3 min).*

In [ ]:
LONG = 10
_, tr, _ = strauss_trace(proc_s, seed=0, n_steps=LONG * steps_default, every=200)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(tr[:, 0], tr[:, 1], lw=0.8, label="n in W⁺ (the chain's state)")
ax.plot(tr[:, 0], tr[:, 2], lw=0.8, label="n in W (what gets saved)")
ax.axvline(steps_default, color="k", ls="--", label=f"default stop ({steps_default} steps)")
tail = tr[len(tr) // 2:, 2]
ax.axhspan(tail.mean() - tail.std(), tail.mean() + tail.std(), color="C1", alpha=0.15, label="long-run mean ± sd (n in W)")
ax.set_xscale("symlog", linthresh=1000); ax.set_xlabel("MH step"); ax.set_ylabel("points")
ax.set_title(f"Strauss chain, β={strong['beta']:.0f}, γ={strong['gamma']}, R={strong['radius']}: starts over-dense at Poisson(β), drains", fontsize=9)
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
def close_pairs(p, r):
    return len(cKDTree(p).query_pairs(r)) if len(p) > 1 else 0

SETTINGS = {
    "weak   (β=400, γ=0.8,  R=0.02)": dict(beta=400.0, gamma=0.8, radius=0.02),
    "medium (β=600, γ=0.4,  R=0.03)": dict(beta=600.0, gamma=0.4, radius=0.03),
    "strong (β=900, γ=0.05, R=0.05)": strong,
}
N_SEEDS = 8
t0 = time.time()
print(f"{'setting':32s} {'steps':>7s} | {'n default':>12s} {'n 10x':>12s} | {'S_R default':>12s} {'S_R 10x':>12s} | n/β")
for lab, p in SETTINGS.items():
    base = StraussProcess(**p)
    n0 = _adaptive_steps(None, p["beta"] * WINDOW.expanded(2 * p["radius"]).volume)
    longp = StraussProcess(**p, n_steps=LONG * n0)
    A = np.array([(len(q), close_pairs(q, p["radius"])) for q in
                  (base.sample(region=WINDOW, seed=s).points for s in range(N_SEEDS))])
    B = np.array([(len(q), close_pairs(q, p["radius"])) for q in
                  (longp.sample(region=WINDOW, seed=10_000 + s).points for s in range(N_SEEDS))])
    ms = lambda x: f"{x.mean():6.1f}±{x.std(ddof=1) / np.sqrt(len(x)):4.1f}"
    print(f"{lab:32s} {n0:7d} | {ms(A[:, 0]):>12s} {ms(B[:, 0]):>12s} | {ms(A[:, 1]):>12s} {ms(B[:, 1]):>12s} | {A[:, 0].mean() / p['beta']:.2f}")
print(f"({time.time() - t0:.0f}s)")

**Reading the table.** Values are mean ± standard error over seeds. If the default and 10×
columns agree within about 2 SE, the fixed budget is adequate at that corner. A systematic
excess of $n$ or $S_R$ in the default column would mean the chain is stopped while still
draining its over-dense start. That error would then be baked into every stored Strauss cloud
in that region of the design. The last column shows how far the realised intensity sits below
the activity $\beta$.

*Not covered here:* LGCP and LGCP–Strauss (`cox.py`, `gibbs.LGCPStraussProcess`) exist only
for the Vihrs comparisons and are excluded from classification. The LGCP thins a dominating
Poisson process by $\exp(\mu + Y(u))$, where $Y$ is a Gaussian random field built from 512 random
Fourier features.

### 4.6 What each knob does

One knob at a time, the others held at the **median of the accepted design**. The knob moves
through its 5th, 50th and 95th design percentiles. Every panel in a gallery uses the **same
seed** (common random numbers), so the differences come from the knob rather than from the
noise. Panels outside the design (rejected by a constraint) are marked in red.

In [ ]:
def sampling_keys(name):
    return list(process_block(name)["design"]["random"]["ranges"])

def knob_gallery(name, seed=0, s=2):
    rnd = process_block(name)["design"]["random"]
    keys = sampling_keys(name)
    V = DESIGNS[name].train_test_vectors
    X = np.array([[v[k] for k in keys] for v in V])
    base = dict(zip(keys, np.median(X, axis=0)))
    qs = np.percentile(X, [5, 50, 95], axis=0)
    fig, axes = plt.subplots(len(keys), 3, figsize=(8.4, 2.9 * len(keys)))
    for i, k in enumerate(keys):
        for j in range(3):
            u = dict(base); u[k] = float(qs[j, i])
            theta = apply_derived(u, rnd.get("derived"))
            ok = passes(theta, rnd.get("constraints"))
            pts = make_process(name, theta).sample(region=WINDOW, seed=seed).points
            ax = axes[i, j]
            show_cloud(ax, pts, f"{k} = {u[k]:.3g}   (n={len(pts)})" + ("" if ok else "\nOUTSIDE DESIGN"), s=s)
            if not ok:
                ax.title.set_color("r")
    fig.suptitle(f"{name}: one knob per row (5th / 50th / 95th design percentile)", y=1.0)
    plt.tight_layout(); plt.show()

knob_gallery("thomas")

In [ ]:
knob_gallery("nested_thomas", s=1.5)

In [ ]:
knob_gallery("strauss")

## 5. Where the design puts its clouds: parameter space in 3-D

Each dot below is **one generated cloud**, placed at its true parameters: the vector
$(\kappa,\mu,\sigma)$ the network is trained to recover. Blue dots are train/test clouds and
orange dots are the adversarial holdout.

**It is a solid, not a surface.** Three parameters are sampled freely, so the clouds fill a
3-D *region*. A surface would only appear if one parameter were tied to the other two. The
region's shape is still exactly predictable. In **log coordinates** the derivation is linear:

$$\log\kappa = \log K,\qquad \log\mu = \log EN - \log K,\qquad \log\sigma = \log c - \log 2 - \tfrac12\log K,$$

so the log-uniform sampling box in $(K, EN, c)$ maps to a **sheared box (a parallelepiped)**,
with the slab $\mu<2.5$ sliced off by a plane. The black wireframe is the image of the sampling
box's 12 edges, computed by pushing points along each edge through the config's own `derived:`
expressions. Edge segments that violate a constraint are dotted grey. The red polygon is the
cut face $\mu = 2.5$. An axis shows $\log_{10}$ of its parameter whenever training uses that
target in log form (`log_label_names`). Otherwise the axis is linear.

In [ ]:
def design_edges(name, n=80):
    """Edges of the sampling box -> target space. Returns a list of (T, ok) per edge,
    T = (n, 3) target values, ok = constraint mask; plus the crossing points of the
    constraint boundary with the box edges (vertices of the cut face)."""
    rnd = process_block(name)["design"]["random"]
    keys, spec = list(rnd["ranges"]), rnd["ranges"]
    tnames, _ = targets_of(name)
    def at(k, s):
        lo, hi = float(spec[k]["low"]), float(spec[k]["high"])
        return math.exp(math.log(lo) + s * (math.log(hi) - math.log(lo))) if spec[k].get("scale", "linear") == "log" else lo + s * (hi - lo)
    def target(u):
        th = apply_derived(u, rnd.get("derived"))
        return [make_process(name, th).params[t] for t in tnames], passes(th, rnd.get("constraints"))
    edges, cuts = [], []
    for free in range(len(keys)):
        others = [j for j in range(len(keys)) if j != free]
        for corner in itertools.product([0.0, 1.0], repeat=len(others)):
            def u_of(s):
                u = {keys[j]: at(keys[j], c) for j, c in zip(others, corner)}
                u[keys[free]] = at(keys[free], s)
                return u
            ss = np.linspace(0, 1, n)
            res = [target(u_of(s)) for s in ss]
            T = np.array([r[0] for r in res]); ok = np.array([r[1] for r in res])
            edges.append((T, ok))
            for a in np.where(ok[:-1] != ok[1:])[0]:           # bisect the boundary crossing
                lo, hi = ss[a], ss[a + 1]
                for _ in range(40):
                    mid = 0.5 * (lo + hi)
                    lo, hi = (mid, hi) if target(u_of(mid))[1] == ok[a] else (lo, mid)
                cuts.append(target(u_of(0.5 * (lo + hi)))[0])
    return edges, np.array(cuts)

def target_array(name, vectors):
    tnames, _ = targets_of(name)
    return np.array([[make_process(name, v).params[t] for t in tnames] for v in vectors])

def ordered_polygon(P):
    """Order coplanar 3-D points around their centroid (via the plane's 2-D PCA basis)."""
    c = P.mean(0); _, _, Vt = np.linalg.svd(P - c)
    xy = (P - c) @ Vt[:2].T
    return P[np.argsort(np.arctan2(xy[:, 1], xy[:, 0]))]

def plot_design_solid(name, views=((22, -60), (5, -100), (60, -20)), log_axes=None):
    tnames, logset = targets_of(name)
    log_axes = [t in logset for t in tnames] if log_axes is None else log_axes
    f = lambda A: np.column_stack([np.log10(A[:, j]) if log_axes[j] else A[:, j] for j in range(A.shape[1])])
    d = DESIGNS[name]
    Xt, Xa = f(target_array(name, d.train_test_vectors)), f(target_array(name, d.adversarial_vectors))
    edges, cuts = design_edges(name)
    labels = [("log₁₀ " if lg else "") + t for t, lg in zip(tnames, log_axes)]
    fig = plt.figure(figsize=(15, 5.2))
    for k, (el, az) in enumerate(views):
        ax = fig.add_subplot(1, len(views), k + 1, projection="3d")
        ax.scatter(*Xt.T, s=0.6, alpha=0.25, color="C0", rasterized=True)
        ax.scatter(*Xa.T, s=1.2, alpha=0.6, color="C1", rasterized=True)
        for T, ok in edges:
            T = f(T)
            Tok = np.where(ok[:, None], T, np.nan); Tno = np.where(~ok[:, None], T, np.nan)
            ax.plot(*Tok.T, color="k", lw=1.0); ax.plot(*Tno.T, color="0.6", lw=0.8, ls=":")
        if len(cuts) >= 3:
            poly = ordered_polygon(f(cuts)); poly = np.vstack([poly, poly[:1]])
            ax.plot(*poly.T, color="r", lw=1.5)
        ax.set_xlabel(labels[0], fontsize=8); ax.set_ylabel(labels[1], fontsize=8); ax.set_zlabel(labels[2], fontsize=8)
        ax.tick_params(labelsize=6); ax.view_init(elev=el, azim=az)
    fig.suptitle(f"{name}: {len(Xt)} train/test (blue) + {len(Xa)} adversarial (orange) clouds in target space", y=0.98)
    plt.tight_layout(); plt.show()

    # the same solid, flattened onto its three coordinate planes
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
    for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
        ax.scatter(Xt[:, i], Xt[:, j], s=0.5, alpha=0.25, color="C0", rasterized=True)
        ax.scatter(Xa[:, i], Xa[:, j], s=0.8, alpha=0.5, color="C1", rasterized=True)
        for T, ok in edges:
            T = f(T); ax.plot(np.where(ok, T[:, i], np.nan), np.where(ok, T[:, j], np.nan), color="k", lw=0.6)
        ax.set_xlabel(labels[i]); ax.set_ylabel(labels[j])
    plt.tight_layout(); plt.show()
    return Xt, Xa, edges, cuts, labels

### 5.1 Thomas: $(\kappa, \mu, \sigma)$

What to look for:
* The solid is **tilted**. Large $\kappa$ goes with small $\sigma$ (slope $-\tfrac12$ in log–log)
  and with small $\mu$ (slope $-1$), because $c$ and $EN$ are held in fixed ranges. **The
  targets are correlated before any data is seen** (§5.5 quantifies this).
* The **red face** is the $\mu = 2.5$ cut. It removes the large-$\kappa$, small-$\mu$ edge.
* The orange adversarial clouds are spread **through the same solid**. They are not at its edge
  or outside it (§6.2).

In [ ]:
THOMAS_SOLID = plot_design_solid("thomas")

### 5.2 Matérn cluster: $(\kappa, \mu, R)$

This is the same design as Thomas, and in fact the **same 8000 $(K, EN, c)$ vectors** (§6.3).
$R = c/\sqrt K = 2\sigma$, so the solid is the Thomas solid shifted up by $\log_{10}2$ along the
third axis.

In [ ]:
MATERN_SOLID = plot_design_solid("matern_cluster")

### 5.3 Strauss: $(\beta, \gamma, R)$

This design is a plain box: log-uniform $\beta$, uniform $\gamma$ and $R$, with no constraints.
The box is uniform in the *model* parameters, but that says little about what the clouds look
like. The second figure colours each stored cloud by $n/\beta$, the fraction of the activity
that is realised as points. It shows a whole region (small $\gamma$, large $R$) where the
realised intensity is a fraction of $\beta$. In that region, large changes in $\beta$ make only
small changes to the cloud, which makes $\beta$ hard to estimate there.

In [ ]:
STRAUSS_SOLID = plot_design_solid("strauss", log_axes=[True, False, False])

recs = load_records("strauss")
if recs is not None:
    B = np.array([[r["params"]["beta"], r["params"]["gamma"], r["params"]["radius"], r["n_points"]] for r in recs])
    fig = plt.figure(figsize=(13, 5))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    sc = ax.scatter(np.log10(B[:, 0]), B[:, 1], B[:, 2], c=B[:, 3] / B[:, 0], s=1, cmap="viridis", rasterized=True)
    ax.set_xlabel("log₁₀ β"); ax.set_ylabel("γ"); ax.set_zlabel("R"); ax.view_init(20, -50)
    fig.colorbar(sc, ax=ax, shrink=0.6, label="n / β")
    ax = fig.add_subplot(1, 2, 2)
    sc = ax.scatter(B[:, 1], B[:, 2], c=B[:, 3] / B[:, 0], s=2, cmap="viridis", rasterized=True)
    ax.set_xlabel("γ"); ax.set_ylabel("R"); fig.colorbar(sc, ax=ax, label="n / β")
    ax.set_title("realised fraction of the activity β")
    plt.tight_layout(); plt.show()

### 5.4 Interactive version

If `plotly` is installed, this cell draws the same solids rotatably, in the same coordinates. Plotly
is not in `cloud-env` (`pip install plotly` adds it). Without it the cell is skipped.

In [ ]:
def plotly_solid(name, solid):
    Xt, Xa, edges, cuts, labels = solid
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=Xt[:, 0], y=Xt[:, 1], z=Xt[:, 2], mode="markers",
                               marker=dict(size=1.5, color="#1f77b4", opacity=0.35), name="train/test"))
    fig.add_trace(go.Scatter3d(x=Xa[:, 0], y=Xa[:, 1], z=Xa[:, 2], mode="markers",
                               marker=dict(size=2, color="#ff7f0e", opacity=0.7), name="adversarial"))
    tnames, _ = targets_of(name)
    to_plot = lambda T: np.column_stack([T[:, j] if not labels[j].startswith("log") else np.log10(T[:, j]) for j in range(3)])
    xs, ys, zs = [], [], []
    for T, ok in edges:
        T = to_plot(T); T[~ok] = np.nan
        xs += list(T[:, 0]) + [None]; ys += list(T[:, 1]) + [None]; zs += list(T[:, 2]) + [None]
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="black", width=3), name="design box edges"))
    if len(cuts) >= 3:
        poly = ordered_polygon(to_plot(cuts)); poly = np.vstack([poly, poly[:1]])
        fig.add_trace(go.Scatter3d(x=poly[:, 0], y=poly[:, 1], z=poly[:, 2], mode="lines",
                                   line=dict(color="red", width=5), name="constraint cut"))
    fig.update_layout(title=name, height=650, scene=dict(xaxis_title=labels[0], yaxis_title=labels[1], zaxis_title=labels[2]))
    fig.show()

if HAVE_PLOTLY:
    for nm, sol in [("thomas", THOMAS_SOLID), ("matern_cluster", MATERN_SOLID), ("strauss", STRAUSS_SOLID)]:
        plotly_solid(nm, sol)
else:
    print("plotly not installed -- skipping interactive plots")

### 5.5 Five-parameter processes: corner plots

Nested Thomas and anisotropic Thomas have five targets each, too many for one 3-D picture. A
corner plot shows every pairwise projection of the 5-D solid, with marginals on the diagonal.
In the nested plot, the $3 \le \sigma_1/\sigma_2 \le 12$ constraint appears as a **diagonal band**
in the (meta_cluster_scale, cluster_scale) panel. The $\mu \ge 2.5$ cut appears as sharp edges
in the panels involving mean_offspring. A 3-D view of three of the nested targets follows.

In [ ]:
def corner_plot(name, s=0.4):
    tnames, logset = targets_of(name)
    d = DESIGNS[name]
    A = target_array(name, d.train_test_vectors)
    X = np.column_stack([np.log10(A[:, j]) if t in logset else A[:, j] for j, t in enumerate(tnames)])
    labels = [("log₁₀ " if t in logset else "") + t for t in tnames]
    k = len(tnames)
    fig, axes = plt.subplots(k, k, figsize=(2.3 * k, 2.3 * k))
    for i in range(k):
        for j in range(k):
            ax = axes[i, j]
            if j > i:
                ax.axis("off"); continue
            if i == j:
                ax.hist(X[:, i], bins=40, color="0.45")
            else:
                ax.scatter(X[:, j], X[:, i], s=s, alpha=0.25, color="C0", rasterized=True)
            ax.tick_params(labelsize=6)
            if i == k - 1: ax.set_xlabel(labels[j], fontsize=7)
            if j == 0 and i > 0: ax.set_ylabel(labels[i], fontsize=7)
    fig.suptitle(f"{name}: pairwise projections of the {k}-D design ({len(X)} clouds)", y=1.0)
    plt.tight_layout(); plt.show()
    return X, labels

NESTED_X, NESTED_LABELS = corner_plot("nested_thomas")

In [ ]:
fig = plt.figure(figsize=(14, 5))
for k, (cols, view) in enumerate([((0, 1, 3), (20, -60)), ((0, 2, 4), (20, -60))]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.scatter(*NESTED_X[:, cols].T, s=0.5, alpha=0.25, color="C0", rasterized=True)
    ax.set_xlabel(NESTED_LABELS[cols[0]], fontsize=7); ax.set_ylabel(NESTED_LABELS[cols[1]], fontsize=7)
    ax.set_zlabel(NESTED_LABELS[cols[2]], fontsize=7); ax.tick_params(labelsize=6); ax.view_init(*view)
fig.suptitle("nested Thomas, two 3-D slices: counts (κ, μ₁, μ) and scales (κ, σ₁, σ₂)")
plt.tight_layout(); plt.show()

In [ ]:
ANISO_X, ANISO_LABELS = corner_plot("aniso_thomas")

### 5.6 The design builds correlations into the targets

The network is trained on z-scored (log-)targets. The table gives, for each target, the $R^2$ of
a *linear* regression on the **other targets**, computed on the design alone, with no cloud
involved. A high value means the design ties that target to the others. For example, in
Thomas, $\log\mu = \log EN - \log\kappa$, and $EN$ spans a narrower log-range than $K$, so
$\kappa$ largely determines $\mu$.

This matters when reading per-target results. Errors on $\kappa$ and $\mu$ are **not
independent**. $n \approx \kappa\mu$ is nearly observed, so an overestimate of one tends to come
with an underestimate of the other. The chance level ($\approx 1$) quoted in the README is a
label shuffle, which is a legitimate floor, but a network gets part of some targets from the
others "for free" through these correlations.

In [ ]:
def design_r2(X):
    out = []
    for j in range(X.shape[1]):
        Aj = np.column_stack([np.ones(len(X)), np.delete(X, j, axis=1)])
        coef, *_ = np.linalg.lstsq(Aj, X[:, j], rcond=None)
        out.append(1 - np.var(X[:, j] - Aj @ coef) / np.var(X[:, j]))
    return out

for name in CONFIGS:
    tnames, logset = targets_of(name)
    A = target_array(name, DESIGNS[name].train_test_vectors)
    X = np.column_stack([np.log(A[:, j]) if t in logset else A[:, j] for j, t in enumerate(tnames)])
    C = np.corrcoef(X.T)
    print(f"\n{name}")
    print(" " * 18 + " ".join(f"{t[:15]:>15s}" for t in tnames))
    for t, row in zip(tnames, C):
        print(f"   {t[:14]:>14s} " + " ".join(f"{v:+15.2f}" for v in row))
    print("   R² from the others:  " + ",  ".join(f"{t}={r:.2f}" for t, r in zip(tnames, design_r2(X))))

## 6. Stage C: seeds, holdout, and what ends up on disk

### 6.1 The seed scheme

| random stream | seed | used for |
|---|---|---|
| design RNG | `default_rng(0)` | drawing all 8000 parameter vectors |
| holdout RNG | `default_rng(0 + 100000)` | choosing which 1000 vectors are "adversarial" |
| train/test cloud *i* | `default_rng(i)`, i = 0…6999 | simulating cloud *i* |
| adversarial cloud *j* | `default_rng(100000 + j)` | simulating adversarial cloud *j* |
| classification bundle | `source_dir_index · 10⁶ + seed` | only a relabelling for the seed join, no new randomness |

**Every process uses the same seeds.** Thomas cloud 17, Matérn cloud 17, nested Thomas cloud 17
and Strauss cloud 17 are all simulated from `default_rng(17)`. Two streams also collide: the
design RNG is the same stream as train cloud 0, and the holdout RNG is the same stream as
adversarial cloud 0. Those collisions are cosmetic (they affect one cloud each). The first
point has real consequences, examined in §6.3.

In [ ]:
for name in CONFIGS:
    for adv in (False, True):
        recs = load_records(name, adversarial=adv)
        if recs is None:
            continue
        s = [r["seed"] for r in recs]
        print(f"{name:15s} {'adversarial' if adv else 'train/test ':11s} seeds {s[0]} … {s[-1]}  (n = {len(s)})")

### 6.2 The "adversarial" split is a second i.i.d. test set, not an out-of-distribution one

`split_adversarial_vectors` removes a **uniformly random** 12.5% of the parameter vectors. With a
*random* design and `reps: 1`, every parameter vector is already unique, so an ordinary test
cloud also has parameters the model never saw. The adversarial clouds come from exactly the
same distribution as the test clouds, and they are **no further** from the training vectors.

Two checks: (a) a KS test per sampling coordinate, adversarial versus train/test; (b) the
distance from each point to its nearest training vector, in standardised sampling coordinates,
for a random 1000 train/test vectors (standing in for a test split) and for the adversarial
vectors. If the holdout were out of distribution, (b) would be shifted right.

For a real stress test, hold out a *region* instead: for example the top decile of $c$ (the
near-Poisson end) or one corner of the solid in §5. Accuracy on that region measures
extrapolation.

In [ ]:
def sampling_coords(name, vectors):
    spec = process_block(name)["design"]["random"]["ranges"]
    return np.array([[math.log(v[k]) if spec[k].get("scale", "linear") == "log" else v[k] for k in spec] for v in vectors])

fig, axes = plt.subplots(1, len(CONFIGS), figsize=(3.2 * len(CONFIGS), 3))
rng = np.random.default_rng(0)
for ax, name in zip(axes, CONFIGS):
    d = DESIGNS[name]
    tt, adv = sampling_coords(name, d.train_test_vectors), sampling_coords(name, d.adversarial_vectors)
    ks_p = [ks_2samp(tt[:, j], adv[:, j]).pvalue for j in range(tt.shape[1])]
    perm = rng.permutation(len(tt)); test, train = tt[perm[:1000]], tt[perm[1000:]]
    mu_, sd_ = train.mean(0), train.std(0)
    tree = cKDTree((train - mu_) / sd_)
    d_test, d_adv = tree.query((test - mu_) / sd_)[0], tree.query((adv - mu_) / sd_)[0]
    bins = np.linspace(0, max(d_test.max(), d_adv.max()), 30)
    ax.hist(d_test, bins=bins, density=True, histtype="step", label="random test subset")
    ax.hist(d_adv, bins=bins, density=True, histtype="step", label="adversarial")
    ax.set_title(f"{name}\nKS p (per coord): {', '.join(f'{p:.2f}' for p in ks_p)}", fontsize=8)
    ax.set_xlabel("distance to nearest train vector", fontsize=8)
    print(f"{name:15s} median NN distance  test {np.median(d_test):.3f}   adversarial {np.median(d_adv):.3f}")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 6.3 Cross-process coupling: same θ, same seed, same parent layout

Two facts combine here:

1. **Thomas and Matérn cluster have identical design blocks** (same ranges, same derived
   quantities up to $R = 2\sigma$, same `seed: 0`). So they draw **the same 8000 $(K, EN, c)$
   vectors**, in the same order, with the same holdout. Anisotropic Thomas has two extra axes,
   which interleave the draws, so its vectors differ.
2. **Cloud *i* of every process uses `default_rng(i)`.** In `NeymanScottProcess._sample_points`
   the first draw is the parent count (`rng.poisson`) and the second is the parent positions
   (`rng.uniform`). With equal or similar $\kappa|W^+|$, the Poisson draw usually consumes the same
   random numbers. The parent positions are then **the same uniforms**, rescaled to boxes of
   slightly different size.

The expected result: Thomas cloud *i* and Matérn cloud *i* have **their clusters in nearly the
same places**, and clouds from other Neyman–Scott families share much of their parent layout
too. The cells below measure this against an **independent control**, the same parameters with
the seed shifted by 10⁶.

In [ ]:
T, M, A_ = DESIGNS["thomas"], DESIGNS["matern_cluster"], DESIGNS["aniso_thomas"]
same_theta = all(a[k] == b[k] for a, b in zip(T.train_test_design + T.adversarial_design,
                                              M.train_test_design + M.adversarial_design) for k in ("K", "EN", "c"))
print("Thomas and Matérn share every (K, EN, c) vector exactly:", same_theta)
print("... and the same adversarial indices:", T.adversarial_indices == M.adversarial_indices)
print("aniso_thomas vector 0 vs thomas vector 0 (K):", A_.train_test_vectors[0]["K"], T.train_test_vectors[0]["K"],
      "| vector 1:", round(A_.train_test_vectors[1]["K"], 3), round(T.train_test_vectors[1]["K"], 3))

def matched_fraction(P, Q, delta):
    """Fraction of points of P (inside W) with a point of Q within delta."""
    P = P[WINDOW.contains(P)] if len(P) else P
    if len(P) == 0 or len(Q) == 0:
        return np.nan
    return float(np.mean(cKDTree(Q).query(P)[0] <= delta))

def parents_of(name, theta, seed):
    st = steps(make_process(name, theta), seed)
    return st["meta"]["parents"] if "meta" in st else st["parents"]      # nested: grandparents

N_PAIRS = 300
pairs = [("thomas", "matern_cluster"), ("thomas", "aniso_thomas"), ("thomas", "nested_thomas")]
rows = {p: {"coupled": [], "control": [], "same_count": []} for p in pairs}
for i in range(N_PAIRS):
    th_T = T.train_test_design[i]
    PT = parents_of("thomas", th_T, i)
    delta = 0.25 / math.sqrt(th_T["parent_intensity"])        # a quarter of the parent spacing
    for a, b in pairs:
        th_b = DESIGNS[b].train_test_design[i]
        Pb, Pb_ctrl = parents_of(b, th_b, i), parents_of(b, th_b, i + 10**6)
        rows[(a, b)]["coupled"].append(matched_fraction(PT, Pb, delta))
        rows[(a, b)]["control"].append(matched_fraction(PT, Pb_ctrl, delta))
        rows[(a, b)]["same_count"].append(len(PT) == len(Pb))

print(f"\nfraction of Thomas parents with a partner-process parent within ¼ parent-spacing ({N_PAIRS} clouds)")
for (a, b), r in rows.items():
    print(f"  {a} vs {b:15s}  same seed: {np.nanmean(r['coupled']):.2f}   independent seed: {np.nanmean(r['control']):.2f}"
          f"   identical parent count: {np.mean(r['same_count']):.2f}")

In [ ]:
i = 5
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
thT, thM = T.train_test_design[i], M.train_test_design[i]
stT, stM, stC = steps(make_process("thomas", thT), i), steps(make_process("matern_cluster", thM), i), \
    steps(make_process("matern_cluster", thM), i + 10**6)
show_cloud(axes[0], stT["points"], f"Thomas, cloud {i} (seed {i})", s=3, color="C0")
show_cloud(axes[1], stM["points"], f"Matérn cluster, cloud {i} (seed {i})", s=3, color="C3")
ax = axes[2]
ax.scatter(*stT["parents"].T, marker="x", color="C0", label="Thomas parents (seed i)")
ax.scatter(*stM["parents"].T, marker="+", s=70, color="C3", label="Matérn parents (seed i)")
ax.scatter(*stC["parents"].T, marker=".", color="0.6", label="Matérn parents (seed i+10⁶)")
ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, ec="k")); ax.set_aspect("equal")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1)); ax.set_title("parent layouts", fontsize=8)
plt.tight_layout(); plt.show()

**The same coupling in the stored data.** If clouds were independent given θ, the relative
count residual $n_i/EN_i$ of Thomas cloud *i* would be uncorrelated with that of Matérn cloud *i*.
The permuted pairing is the null reference.

In [ ]:
rT, rM = load_records("thomas"), load_records("matern_cluster")
if rT is not None and rM is not None:
    EN_ = np.array([r["params"]["parent_intensity"] * r["params"]["mean_offspring"] for r in rT])
    zT = np.array([r["n_points"] for r in rT]) / EN_
    zM = np.array([r["n_points"] for r in rM]) / EN_
    perm = np.random.default_rng(0).permutation(len(zM))
    print(f"Spearman corr of n/EN, Thomas i vs Matérn i:          {spearmanr(zT, zM)[0]:+.3f}")
    print(f"Spearman corr of n/EN, Thomas i vs Matérn (permuted): {spearmanr(zT, zM[perm])[0]:+.3f}")

**How much this matters.** It does not affect the per-process parameter-estimation experiments,
which never mix processes. It does affect **classification**. There, Thomas cloud *i* (label
`thomas`) and Matérn cloud *i* (label `matern_cluster`) have the same θ and, if the numbers above
confirm it, much the same cluster centres. The random split scatters the two twins
independently, so the twin of a test cloud is often in training *with the other label*. This
does not leak the test label. Identical θ across the two classes is arguably a clean design,
because θ cannot be used as a class cue. But the classes are **not independent samples**, and it
is an open question whether paired near-duplicates with opposite labels help or hurt the
Matérn-vs-Thomas confusion (Matérn recall is the weakest class, 0.64). The idiomatic fix is to
derive each cloud's stream as `SeedSequence([seed, process_id, i])`, which removes the shared
layout and keeps reproducibility.

### 6.4 Point counts

$n$ is the simplest feature there is. The first plot shows how well $EN$ predicts $n$ (the
over-dispersion from §3). The second shows the $n$ distribution per classification class. The
three cluster families share the same $EN$ range by construction. Strauss gets its $n$ from
$\beta$ and the inhibition, so its distribution has a different shape. The repo's `logn_only`
experiment measures how far $n$ alone gets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if rT is not None:
    nT = np.array([r["n_points"] for r in rT])
    axes[0].scatter(EN_, nT, s=1, alpha=0.3, rasterized=True)
    axes[0].plot([150, 800], [150, 800], "r-", lw=1, label="n = EN")
    axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].set_xlabel("EN = κμ"); axes[0].set_ylabel("n")
    axes[0].set_title(f"Thomas: n vs EN   (R² of log n on log EN = {np.corrcoef(np.log(EN_), np.log(nT))[0, 1] ** 2:.2f})", fontsize=9)
    axes[0].legend(fontsize=7)
bins = np.geomspace(40, 1500, 50)
for name in ["thomas", "aniso_thomas", "matern_cluster", "nested_thomas", "strauss"]:
    recs = load_records(name)
    if recs is not None:
        axes[1].hist([r["n_points"] for r in recs], bins=bins, histtype="step", lw=1.3, label=name)
axes[1].set_xscale("log"); axes[1].set_xlabel("n points"); axes[1].legend(fontsize=7)
axes[1].set_title("n per source process (aniso_thomas is merged into 'thomas' for classification)", fontsize=9)
plt.tight_layout(); plt.show()

man = DATA / "classification" / "classification_manifest.yaml"
if man.exists():
    m = yaml.safe_load(open(man))
    cc = m["artefacts"]["clouds.pkl"]["class_counts"]
    print("classification class counts:", cc, "| majority-class accuracy:", round(max(cc.values()) / sum(cc.values()), 3))
    print("merged labels:", m.get("merged_labels"), "| excluded:", m.get("excluded_processes"))

## 7. Scrutiny summary

**What checks out**

* **Reproducibility.** The designs rebuild exactly from the configs. Every stored target
  parameter and seed matches, and re-simulated clouds are bit-identical to the stored ones (§2.2).
* **The simulators do what they claim.** The step-by-step replicas of the Neyman–Scott engine
  (flat and nested) and of the Strauss birth–death sampler reproduce the library bit for bit
  (§3.1, §4.4, §4.5). The Strauss acceptance ratios are the standard Geyer–Møller ones.
* **Edge effects are handled.** The 3.72σ buffer (exact radius for the disk kernel) gives a
  flat intensity profile and $\mathbb E[n] = EN$. Removing it produces a visible edge deficit (§3.2).
* **The kernels are width-matched.** At equal $c$, Thomas, Matérn and anisotropic Thomas have
  the same RMS cluster width, so the families differ in cluster *shape* only (§4.1).

**Findings** (✔ = verified from code or manifests while writing this notebook; ▶ = measured when
you run the cell referenced)

| # | finding | where | weight |
|---|---|---|---|
| 1 | ✔ The **"adversarial" split is an i.i.d. holdout from the same random design**, not an out-of-distribution one. With `reps: 1`, ordinary test clouds already have unseen θ. "−0.3 points on the adversarial split" is a second test-set estimate, not evidence of robustness. The README's "drawn off the training design grid" overstates it. | §6.2 | affects how results are **interpreted** |
| 2 | ✔ **Thomas and Matérn cluster share all 8000 θ vectors and all seeds**. ▶ Because every process uses `default_rng(i)` for cloud *i*, Neyman–Scott clouds with the same index share much of their **parent layout**. In classification the classes are therefore paired near-twins with opposite labels, not independent samples. | §6.3 | classification only; direction of the effect unknown |
| 3 | ▶ The **Strauss sampler runs a fixed budget** (~40 proposals per expected point) from an over-dense Poisson(β) start, with no convergence diagnostic. The 10×-chain comparison shows whether that is enough in the strong-inhibition corner. | §4.5 | Strauss data quality |
| 4 | ✔ **The targets are correlated by construction**: $\log\mu = \log EN - \log\kappa$, $\log\sigma = \log c - \tfrac12\log\kappa + \text{const}$, plus the μ ≥ 2.5 cut. Per-target errors are not independent, and part of some targets is implied by others. | §5.5, §5.6 | interpretation of per-target losses |
| 5 | ✔ **Strauss β is an activity, not an intensity.** Where γ is small and R is large, the realised intensity is far below β, so the design is uniform in a parameter the clouds are only weakly sensitive to in that region. | §4.5, §5.3 | explains part of Strauss's difficulty |
| 6 | ✔ **Latent config trap**: `configs/runs/nested_thomas/nested_thomas_vihrs.yaml` has a *different* `process.design` (raw-parameter ranges, 2000 × 4 reps) but writes to the same `data/nested_thomas/`. It is inert today, because `generate.py` skips when `clouds.pkl` exists and the manifest's 7000 train vectors rule out the 1750-vector vihrs design. But `generate.py nested_thomas_vihrs.yaml --force` would silently swap the dataset. `thomas/mincontrast*.yaml` have no `adversarial:` block and would regenerate with no holdout. `ProcessConfig.params` is never passed to `CloudDesign`. | §0 | latent |
| 7 | ✔ **RNG stream reuse**: design RNG = train cloud 0's stream, and holdout RNG = adversarial cloud 0's stream. | §6.1 | cosmetic |
| 8 | ✔ **Naming**: for nested Thomas, the target `parent_intensity` is the *grandparent* intensity κ. | §4.4 | cosmetic |
| 9 | ✔ **Point counts**: the cluster families share the EN range; Strauss's $n$ distribution has a different shape. $n$ is a class cue that needs no topology. | §6.4 | context for classification claims |

**Suggested fixes, cheapest first:** derive cloud streams from `SeedSequence([seed, process_id, i])`
(removes finding 2 and finding 7). Add a region-holdout split next to the random one (turns
finding 1 into a real extrapolation test). Log a Strauss convergence summary, or scale
`n_steps` with the interaction strength (finding 3). Delete or re-point the stray design blocks
(finding 6).